# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates step-by-step loading, inspection, and basic analysis of the FAIR^2 dataset using the `mlcroissant` library. All data entities—record sets, fields, and columns—are referenced by their Croissant `@id` identifiers for clarity and reproducibility.

### Dataset Source
The dataset is described by a Croissant schema at:

https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Install the mlcroissant library if not already installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and explore high level information using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL for the dataset
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the metadata and show top-level metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # Accessed as object, not dict

print(f"Dataset: {metadata.name}\nDescription: {metadata.description}\nPublished: {getattr(metadata, 'datePublished', 'N/A')}")

## 2. Data Overview
List the available record sets, their `@id`s, and the fields in each record set using their `@id`s. This helps understand what tables and attributes are present before extracting data.

**Note:** All references are to Croissant `@id` fields.

In [ ]:
# Retrieve all record sets and print their @id, fields, and column mappings
record_sets = list(metadata.record_sets)
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"Record Set: {rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Description: {getattr(rs, 'description', '')}")
    print("  Fields (by @id):")
    for f in rs.fields:
        print(f"    - {f.id} ({getattr(f, 'name', 'N/A')})")
    print(f"  Columns (if tabular source)")
    if hasattr(rs, 'columns'):
        for c in rs.columns:
            print(f"    - {c.id} ({c.name})")
    print('-'*40)

## 3. Data Extraction
Extract all records from selected record sets in the dataset and load them as pandas DataFrames for analysis.

Please use the record set and field `@id`s listed above to choose the data to extract. Here, we demonstrate loading each available record set. If there's more than one, all are loaded for interactive exploration.

In [ ]:
# Load all record sets using their @id fields
dataframes = {}
for rs in record_sets:
    records = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(records)
    dataframes[rs.id] = df
    print(f"- Loaded DataFrame for record set @id={rs.id}: shape {df.shape}")
    print(f"  Columns: {list(df.columns)}\n")
# Pick the first available record set for detailed demonstration
selected_record_set_id = record_sets[0].id if record_sets else None
if selected_record_set_id is not None:
    print(f"Preview of data from record set {selected_record_set_id}:")
    display(dataframes[selected_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
You can now apply operations such as filtering by a numeric field, normalization, and grouping. Please reference the record set and field identifiers found above for operations.

Below, we select a numeric field (by `@id`) for EDA. Please update `numeric_field_id` and `group_field_id` to the appropriate `@id` from your dataset (see section 2 outputs above).

In [ ]:
import numpy as np

# --- Customize these fields based on outputs from Section 2 ---
# Example field IDs from the FAIR^2 dataset (replace as needed!):
# Suppose a field: cr:peptideCounts (number of peptides for a protein)
# Suppose a group field: cr:description (protein description)
numeric_field_id = None  # E.g., 'cr:peptideCounts'
group_field_id = None    # E.g., 'cr:description'

# Try to auto-guess numeric field from DataFrame columns
df = dataframes[selected_record_set_id]
for c in df.columns:
    # Heuristic: choose first integer/float-looking column
    if df[c].dtype in [np.int64, np.float64]:
        numeric_field_id = c
        break
# Use a group-able field if available
for c in df.columns:
    # Heuristic: first object (string) column not same as numeric
    if df[c].dtype == object and c != numeric_field_id:
        group_field_id = c
        break

print(f"Using numeric_field_id: {numeric_field_id}")
print(f"Using group_field_id: {group_field_id}\n")

# Only proceed if a numeric field was found
if numeric_field_id is not None:
    threshold = df[numeric_field_id].quantile(0.75)  # Top quartile as example
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered rows with {numeric_field_id} above {threshold}:")
    print(filtered_df[[numeric_field_id]].head())
    
    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered rows:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    
    # Grouping by group_field_id
    if group_field_id in filtered_df.columns:
        grouped = (
            filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            .reset_index()
            .sort_values(numeric_field_id, ascending=False)
        )
        print(f"\nGrouped mean of {numeric_field_id} by {group_field_id} (top 5):")
        print(grouped.head())
else:
    print("No numeric field could be automatically detected for EDA. Please review column names in the previous outputs.")

## 5. Visualization
Visualize distributions or relationships in the data using the selected fields. Here, histograms and boxplots are generated for the chosen numeric field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None:
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, ax=ax[0])
    ax[0].set_title(f"Distribution of {numeric_field_id}")
    sns.boxplot(x=df[numeric_field_id].dropna(), ax=ax[1])
    ax[1].set_title(f"Boxplot of {numeric_field_id}")
    plt.tight_layout()
    plt.show()
    
    if group_field_id is not None:
        plt.figure(figsize=(10,6))
        # For brevity, plot only top 10
        grouped = (
            df.groupby(group_field_id)[numeric_field_id].mean().nlargest(10)
        )
        sns.barplot(x=grouped.values, y=grouped.index)
        plt.title(f"Top 10 {group_field_id} by mean {numeric_field_id}")
        plt.xlabel(f"Mean {numeric_field_id}")
        plt.ylabel(group_field_id)
        plt.show()

## 6. Conclusion

- This notebook illustrated how to load and examine a complex, FAIR-certified dataset using the Croissant specification and the `mlcroissant` library.
- All field operations use explicit `@id` references for clarity and reproducibility.
- Preliminary EDA showed basic filtering, normalization, grouping, and visualization of the data.

**Next steps:**
- Explore additional record sets and their relationships.
- Integrate with downstream analysis, e.g., statistical hypothesis testing or machine learning.
- Consult the [Croissant documentation](https://mlcommons.github.io/croissant/) for advanced querying and data manipulation features.